# Zero-shot classification of textual variants

**Goal:** sort each variant in our corpus into one of three categories (**error**, **substantive**, **vocabulary**) *without training a model ourselves*.

We work in **two steps**, each with the tool that suits it:

1. **Error** → **letter similarity**. A misspelling ("igmmatic" → "enigmatic") shares most of its letters with the correction. We just count.
2. **Vocabulary or substantive** → **a language model**. Here letters do not help ("wood" → "forest" shares almost none): we need meaning.

**Model:** [`facebook/bart-large-mnli`](https://huggingface.co/facebook/bart-large-mnli), a ~400M-parameter language model fine-tuned for *Natural Language Inference* (NLI).


### How does "zero-shot" work?
The model was trained on one task only: given a **premise** and a **hypothesis**, is the hypothesis
*entailed*, *neutral*, or *contradicted*?

We reuse that skill for classification:

| | Example |
|---|---|
| **Premise** (our data) | *In the manuscript, "kign" was replaced with "king".* |
| **Hypotheses** (one per category, written with the two readings) | *"kign" is a misspelling of "king".* / *"king" means the same as "kign".* / *"king" means something different from "kign".* |

The model scores how strongly the premise **entails** each hypothesis. The best-scoring hypothesis gives the predicted category.
"Zero-shot" = the model has never seen our categories; we only describe them in words.

For more info, see [readme](readme.md).

## Load the corpus
The settings below match our CSV file:
- `DEL_COL`: the column with the deleted reading (set to `None` if you don't have one).
- `ADD_COL`: the column with the added reading.
- `GOLD_COL`: a column with the category set manually in your ground truth (used for evaluation here below; otherwise `None`).

In [1]:
import pandas as pd


CSV_PATH = "gt.csv"   # path to the file, can be changed to "gt_fake.csv" for the fake data
SEP      = ","              # use ";" or "\t" depending on how the file is separated
DEL_COL = "del_full"           # or None
ADD_COL = "add_full"
DEL_CONTEXT = "del_in_context"  # or None
ADD_CONTEXT = "add_in_context"  # or None
GOLD_COL = "category"         # or None

try:
    df = pd.read_csv(CSV_PATH, sep=SEP)
    print(f"Loaded {len(df)} variants from {CSV_PATH}")
except FileNotFoundError:
    print("File not found.")

df.head() # display the first few rows of the dataframe (table)

Loaded 31 variants from gt.csv


,mod_id,notebook,folio,line_n,add_hand,word_scope,del_full,add_full,del_in_context,add_in_context,hand,n_removed,n_added,letter_similarity,case_only,same_lemma,category,Notes
0,ox-ms_abinger_c57-0061-mod002,ox-ms_abinger_c57,31r,5,NaN,word,tore,tearing,fled â On seeing me he darted towards me and...,fled â On seeing me he darted towards me and...,not specified,1,1,0.29,0,1,error,NaN
1,ox-ms_abinger_c57-0061-mod004,ox-ms_abinger_c57,31r,22,NaN,word,softness,gentleness,the miserable pain of & The feelings of kindne...,the miserable pain of & The feelings of kindne...,not specified,1,1,0.50,0,0,vocabulary,NaN
2,ox-ms_abinger_c57-0061-mod005,ox-ms_abinger_c57,31r,27,NaN,word,pain,the agony of my wound,vowed eternal hatred & vengeance to all mankin...,vowed eternal hatred & vengeance to all mankin...,not specified,1,5,0.10,0,0,substantive,NaN
3,ox-ms_abinger_c57-0062-mod002,ox-ms_abinger_c57,31v,3,#pbs,word,prayers,vows,of the injustice & ingratitude of their inflic...,of the injustice & ingratitude of their inflic...,#pbs,1,1,0.14,0,0,vocabulary,NaN
4,ox-ms_abinger_c57-0062-mod003,ox-ms_abinger_c57,31v,6,#pbs,word,horrors of my situation.,the outrages & the anguish I had endured.,& deadly revenge such as would alone compensat...,& deadly revenge such as would alone compensat...,#pbs,4,7,0.26,0,0,substantive,NaN


## Step 1. Errors: letter similarity

The neural model BART-NMLI has no notion of spelling: internally, it cuts an unknown word like "igmmatic" into pieces, and cannot tell whether a word exists.

Instead we measure how many letters the two readings share (as in session 6):
1.0 = identical, 0 = nothing in common. Capitals are ignored.

If the similarity is above a threshold, the variant is an **error**. 0.65 worked well on our examples: try other values.

In [2]:
# Levenshtein similarity between two strings can also be used here, as in session 6.

import difflib  # Built into Python: compares two texts letter by letter.

def letter_similarity(a, b):
    # 1.0 = identical, 0.0 = nothing in common. Ignores capitals.
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()

ERROR_THRESHOLD = 0.65

df["similarity"] = [round(letter_similarity(d, a), 2) for d, a in zip(df[DEL_COL], df[ADD_COL])]
df["is_error"] = df["similarity"] >= ERROR_THRESHOLD

print(df["is_error"].sum(), "variants classified as error")
df[[DEL_COL, ADD_COL, "similarity", "is_error"]].sort_values("similarity", ascending=False).head(25)

8 variants classified as error


,del_full,add_full,similarity,is_error
27,The,the,1.00,True
17,portraid,portrait,0.88,True
10,horrow,horror,0.83,True
14,trroat,throat,0.83,True
22,somply,comply,0.83,True
7,gazen,gazed,0.80,True
23,createfor me.,create.,0.70,True
15,clapped,clapping,0.67,True
1,softness,gentleness,0.56,False
19,hided,hiding,0.55,False


## Step 2. Vocabulary or substantive: the language model

The model only looks at the variants that are **not** errors.

### Build the premise
The model reads one sentence per variant: the **premise**.

The premise is part of the prompt, but it’s not necessarily the whole prompt. It’s the sentence or context the model is given; the prompt can also include instructions about what to do with it.

Two choices here:
- **Only the change**, without the surrounding sentence: for this small model, more text means more distraction.
- **A neutral verb**: *replaced*. A verb like *corrected* would already suggest that the deleted reading was an error.

The commented lines are earlier versions, kept for comparison.

In [3]:
def make_premise(row):
    # return f'The original reads "{row[DEL_COL]}" in context "{row[DEL_CONTEXT]}"; the variant reads "{row[ADD_COL]}" in context "{row[ADD_CONTEXT]}".'
    # return f'The original reads "{row[DEL_COL]}"; the variant reads "{row[ADD_COL]}".'
    # return f'"{row[DEL_COL]}" was deleted and "{row[ADD_COL]}" was added.'
    # return f'The difference between "{row[DEL_COL]}" and "{row[ADD_COL]}".'
    return f'In the manuscript, "{row[DEL_COL]}" was replaced with "{row[ADD_COL]}".'

df["premise"] = df.apply(make_premise, axis=1)
df["premise"].iloc[0]

'In the manuscript, "tore" was replaced with "tearing".'

### Write the hypotheses
Each hypothesis is a **claim about the two readings of this variant**, which the premise can support or not:

| Category | Hypothesis |
|---|---|
| vocabulary | *"forest" means something similar to "wood".* |
| substantive | *"forest" means something different from "wood".* |

Because the hypotheses change with every variant, we write a function that builds them.

In [4]:
def make_hypotheses(row):
    # Returns one hypothesis per category, written with the two readings of this variant.
    deleted, added = row[DEL_COL], row[ADD_COL]
    return {
        "vocabulary":  f'"{added}" means something similar to "{deleted}".',
        "substantive": f'"{added}" means something different from "{deleted}".',
    }

for category, hypothesis in make_hypotheses(df.iloc[24]).items():
    print(f"{category:<12} {hypothesis}")

vocabulary   "looks" means something similar to "eyes".
substantive  "looks" means something different from "eyes".


## Load the model and test one example
The first run downloads the model (~1.6 GB). It runs on CPU; a GPU is faster but not required.

In [5]:
from transformers import pipeline


classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

def classify(row):
    # Returns the score of each category for one variant.
    hypotheses = make_hypotheses(row)
    result = classifier(
        row["premise"],
        list(hypotheses.values()),  # The hypotheses are the candidate "labels".
        hypothesis_template="{}",   # They are already full sentences: nothing to add around them.
        multi_label=False,          # The two hypotheses compete: their scores sum to 1.
    )
    score_of = dict(zip(result["labels"], result["scores"]))  # hypothesis -> score
    return {category: round(score_of[h], 3) for category, h in hypotheses.items()}

# Test on two variants: a synonym and a change of meaning.
for i in [12, 25]:
    row = df.iloc[i]
    print("Premise:", row["premise"])
    for category, score in classify(row).items():
        print(f"  {category:<12} {score:.3f}")

/home/elena/Github/DHBern/text-analysis-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4009.56it/s]


Premise: In the manuscript, "eyes" was replaced with "face".
  vocabulary   0.756
  substantive  0.244
Premise: In the manuscript, "an answer." was replaced with "areply.".
  vocabulary   0.671
  substantive  0.329


With `multi_label=False`, the two hypotheses compete: their scores sum to 1. So one number is enough: `score_vocabulary`, how much the model leans towards *same meaning*. (`score_substantive` = 1 − `score_vocabulary`.)

### Score all the variants that are not errors
This takes a few seconds for 60 variants, a few minutes for the whole corpus.

In [6]:
to_score = df[~df["is_error"]]  # ~ means "not".

'''Classify only the variants that are not errors.
scores = pd.DataFrame([classify(row) for _, row in to_score.iterrows()], index=to_score.index)

df["score_vocabulary"] = scores["vocabulary"]  # Empty (NaN) for the errors.

df.loc[~df["is_error"], [DEL_COL, ADD_COL, "score_vocabulary"]].sort_values("score_vocabulary")
'''


scores = pd.DataFrame([classify(row) for _, row in to_score.iterrows()], index=to_score.index)

df["predicted"]  = scores.idxmax(axis=1)  # The category with the highest score.
df["confidence"] = scores.max(axis=1)     # Its score.
for category in scores.columns:
    df["score_" + category] = scores[category]  # Keeps all the scores, to look at them later.

df[[DEL_COL, ADD_COL, "score_vocabulary", "score_substantive", "predicted"]]

,del_full,add_full,score_vocabulary,score_substantive,predicted
0,tore,tearing,0.807,0.193,vocabulary
1,softness,gentleness,0.744,0.256,vocabulary
2,pain,the agony of my wound,0.560,0.440,vocabulary
3,prayers,vows,0.648,0.352,vocabulary
4,horrors of my situation.,the outrages & the anguish I had endured.,0.482,0.518,substantive
5,journey,toils,0.625,0.375,vocabulary
6,Geneva,that town,0.604,0.396,vocabulary
7,gazen,gazed,NaN,NaN,NaN
8,athought,an idea,0.599,0.401,vocabulary
9,hav e,had,0.721,0.279,vocabulary


## Evaluate (only if you have manual labels)
Compare the model with **your** judgement.
- **Accuracy**: share of variants where model and human agree.
- **Confusion matrix**: rows = human label, columns = model label. Off-diagonal cells show *which* categories the model confuses.

In [7]:
if GOLD_COL and GOLD_COL in df.columns:
    accuracy = (df["predicted"] == df[GOLD_COL]).mean()
    print(f"Accuracy: {accuracy:.0%}")
    display(pd.crosstab(df[GOLD_COL], df["predicted"], rownames=["human"], colnames=["model"]))
else:
    print("No manual labels: skipping evaluation.")

Accuracy: 35%


model,substantive,vocabulary
human,,
error,0,5
substantive,1,5
vocabulary,0,10


## Save the results

In [8]:
df.drop(columns="premise").to_csv("variants_classified_zero_shot_except_errors.csv", index=False)
print("Saved to variants_classified_zero_shot_except_errors.csv")

Saved to variants_classified_zero_shot_except_errors.csv


## Things to keep in mind
- **Each tool where it works:** letters for spelling, the model for meaning.
- **Letter similarity has blind spots:** short words change a lot with one letter ("of" → "or"); some synonyms share many letters ("blossomed" → "bloomed").
- **Names:** the model may judge two names ("England" → "Archangel") as "similar", although for us the change is substantive.
- **Prompt sensitivity:** small changes in the premise or the hypotheses can change the results noticeably.
- **Language:** this model was trained on **English**. For other languages, try a multilingual NLI model, e.g. `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` (change only the `model=` name).

### Exercises
1. Change `ERROR_THRESHOLD` (e.g. 0.6, 0.75). Which errors are found or lost? Which other variants become errors?
